# 04 — dPCA, GPFA, RRR, and one common manifold for the three areas

Continues `04_dimred_msr.ipynb` on the same session (`5c0c560e`). Five things:

1. **Bin size x smoothing sweep** — 10 / 20 / 50 ms bins x 0 / 20 / 40 / 80 ms Gaussian
   sigma. Everything downstream depends on this choice, and until now it was simply
   asserted (20 ms, sigma = 40 ms). Smoothing is specified in **milliseconds, not bins**, so
   that changing the bin size does not silently change the smoothing too.
2. **Embedding quality metrics** — variance explained is printed *on the PCA axes*, and every
   method carries what it actually has: MDS stress-1, t-SNE KL, Isomap reconstruction error,
   and — for all of them — trustworthiness. **UMAP, Laplacian eigenmaps and PHATE have no
   native goodness-of-fit scalar.** That is a fact about the methods, not an omission.
3. **dPCA** — splits the variance into *time*, *stimulus*, *decision*, and interactions.
   This is the one that speaks to the project's actual claim.
4. **GPFA** — latent trajectories with an inferred **time prior**, on raw counts. The only
   method here that does not need our smoothing at all, so it is the check on whether the
   trajectories we have been drawing were put there by the smoother.
5. **RRR / communication subspace** — how many dimensions does one area send to another,
   with Semedo's within-area control. Then a **common manifold** across the triplet, with
   each area projected into it.

In [ ]:
import sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.viz.style import set_style, savefig, region_color, stroke, OKABE_ITO, SEQUENTIAL
set_style()

SESSION_EID = "5c0c560e-9e1f-45e9-b66e-e4ee7855be84"
REGIONS = ["VISp", "SCm", "MRN"]
PRE_S, POST_S = 0.5, 0.8
BIN_SIZES_S  = [0.010, 0.020, 0.050]        # the three bin widths to compare
SMOOTH_MS    = [0.0, 20.0, 40.0, 80.0, 160.0]   # Gaussian sigma, in MILLISECONDS
SEED = 0
PHASE = "phase3_manifold"

rng = np.random.default_rng(SEED)
OUT = ROOT / "results" / "manifold"
OUT.mkdir(parents=True, exist_ok=True)
print(f"figures -> results/figures/{PHASE}/   tables -> results/manifold/")

## 1. Load the session and build the tensors at all three bin sizes

In [ ]:
from src.data.access import get_one, load_trials, load_units, sessions_table
from src.data.build_matrices import region_tensor_from_probes, region_spike_trains
from iblatlas.regions import BrainRegions

t0 = time.time()
one = get_one(cache_dir=ROOT / "data" / "ONE")
pids = sessions_table(one).query("eid == @SESSION_EID")["pid"].tolist()

trials, mask = load_trials(one, SESSION_EID)
good = trials[mask]
events = good["stimOn_times"].to_numpy()
rt     = (good["firstMovement_times"] - good["stimOn_times"]).to_numpy()
choice = good["choice"].to_numpy()                      # -1 / +1
stim_right = ~np.isnan(good["contrastRight"].to_numpy())

br = BrainRegions()
probe_loads = []
for pid in pids:
    spikes, clusters = load_units(one, pid)
    clusters = clusters.copy()
    clusters["Beryl"] = br.id2acronym(clusters["atlas_id"].values, mapping="Beryl")
    probe_loads.append({"spikes": spikes, "clusters": clusters})

TASK = (float(events.min() - PRE_S - 1), float(events.max() + POST_S + 1))

# tensors at every bin size + the continuous trains (for GPFA)
tensors = {}     # (region, bin_s) -> (counts, bin_centers)
trains  = {}     # region -> {unit_col: spike_times}
for region in REGIONS:
    trains[region] = region_spike_trains(probe_loads, region, t_start=TASK[0], t_stop=TASK[1])
    for b in BIN_SIZES_S:
        tensors[(region, b)] = region_tensor_from_probes(
            probe_loads, region, events, pre_s=PRE_S, post_s=POST_S, bin_size_s=b)

print(f"\n{len(good)} trials | choice balance {np.mean(choice > 0):.2f} | "
      f"stim-right {stim_right.mean():.2f}")
for region in REGIONS:
    shapes = {f"{int(b*1000)}ms": tensors[(region, b)][0].shape for b in BIN_SIZES_S}
    print(f"[{region}] {shapes}")
print(f"load + bin: {time.time() - t0:.1f} s")

## 2. The sweep: does the geometry survive a change of bin size and smoothing?

Until now `20 ms bins, sigma = 40 ms` was an assumption. It needs to be a finding, because
the CLAUDE.md traps file already records one case where bin size decided the science (the
rSLDS latent is shot-noise at 20 ms and recovers real attractors at 50 ms).

Four numbers per configuration, all held-out where that is meaningful:

* **PC1+2 variance** — how much of the geometry two dimensions can even show.
* **time R²** — Ridge on 10 PCs, `GroupKFold` **by trial**. Can you recover *when in the
  trial* you are? This is the stereotyped-trajectory claim as a number.
* **choice AUC** — logistic regression on the 50–250 ms window.
* **trustworthiness** — of the 2-D PCA map, i.e. how much of the local neighbourhood
  structure a 2-D picture preserves.

A result that only exists at one bin size is not a result.

> **Do not select the smoothing by maximising `time_R2`.** A Gaussian kernel *imposes*
> temporal autocorrelation, and `time_R2` measures how predictable time is from the
> smoothed signal — so maximising it maximises the smoother, and the argmax runs away to
> the largest sigma you happen to have swept. The cell below detects that and refuses the
> argmax. The configuration is chosen on **external** grounds instead (the rSLDS
> ground-truth bin size, and the GPFA timescales actually fitted in §5).

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_score
from sklearn.manifold import trustworthiness
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.geometry.embed import population_matrix

N_PC = 10

def evaluate(counts, bin_centers, bin_s, sigma_ms, seed=SEED):
    n_trials, n_bins, _ = counts.shape
    X = population_matrix(counts, bin_size_s=bin_s, smooth_sigma_ms=sigma_ms)
    t_row = np.tile(bin_centers, n_trials)
    g_row = np.repeat(np.arange(n_trials), n_bins)

    p = PCA(N_PC, random_state=seed).fit(X)
    Xp = p.transform(X)
    r2 = cross_val_score(Ridge(alpha=1.0), Xp, t_row, groups=g_row,
                         cv=GroupKFold(5), scoring="r2").mean()

    w = (bin_centers >= 0.05) & (bin_centers < 0.25)
    R = X.reshape(n_trials, n_bins, -1)[:, w, :].mean(1)          # (trials, units)
    auc = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
                          R, choice, cv=StratifiedKFold(5, shuffle=True, random_state=seed),
                          scoring="roc_auc").mean()

    sub = np.random.default_rng(seed).choice(X.shape[0], min(800, X.shape[0]), replace=False)
    trust = trustworthiness(X[sub], Xp[sub, :2], n_neighbors=15)

    return {"var_pc12": float(p.explained_variance_ratio_[:2].sum()),
            "time_R2": float(r2), "choice_AUC": float(auc), "trust_2d": float(trust)}

t0 = time.time()
rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for region in REGIONS:
        for b in BIN_SIZES_S:
            counts, bc = tensors[(region, b)]
            for s in SMOOTH_MS:
                m = evaluate(counts, bc, b, s)
                rows.append({"region": region, "bin_ms": int(b * 1000), "sigma_ms": s, **m})
        print(f"  {region} done ({time.time() - t0:.0f}s)")

sweep = pd.DataFrame(rows)
sweep.to_csv(OUT / "bin_smooth_sweep.csv", index=False)
print(f"\nsweep: {time.time() - t0:.0f} s")
display(sweep.pivot_table(index=["region", "bin_ms"], columns="sigma_ms",
                          values="time_R2").round(3))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7.5))
metrics = [("time_R2", "time decoding R2 (held-out)"), ("choice_AUC", "choice AUC (held-out)"),
           ("var_pc12", "PC1+2 variance explained"), ("trust_2d", "trustworthiness of 2-D PCA")]
marks = {10: "o", 20: "s", 50: "^"}

for ax, (m, lab) in zip(axes.ravel(), metrics):
    for region in REGIONS:
        for b in [10, 20, 50]:
            d = sweep[(sweep.region == region) & (sweep.bin_ms == b)].sort_values("sigma_ms")
            ax.plot(d["sigma_ms"], d[m], marker=marks[b], color=region_color(region),
                    lw=1.6, ms=5, path_effects=stroke(3.0),
                    label=f"{region} {b}ms" if m == "time_R2" else None)
    ax.set(xlabel="smoothing sigma (ms)", ylabel=lab, title=lab)
axes[0, 0].legend(frameon=False, fontsize=7, ncol=3)
fig.suptitle("Bin size x smoothing sweep — marker = bin (o 10, s 20, ^ 50 ms), colour = region")
fig.tight_layout(); savefig(fig, PHASE, "01_bin_smooth_sweep"); plt.show()

best = (sweep.groupby(["bin_ms", "sigma_ms"])["time_R2"].mean()
              .sort_values(ascending=False).reset_index())
print("mean time_R2 across regions, best configurations first:")
print(best.head(5).to_string(index=False))

# --- do NOT take that argmax. Check first whether it is a boundary. ---
grand = sweep.groupby("sigma_ms")[["time_R2", "choice_AUC", "var_pc12"]].mean().round(3)
print("\nmean over ALL regions and bins, as a function of smoothing:")
print(grand.to_string())

argmax_sigma = float(best.iloc[0]["sigma_ms"])
monotone = grand["time_R2"].is_monotonic_increasing
auc_range = float(grand["choice_AUC"].max() - grand["choice_AUC"].min())

if monotone and argmax_sigma == max(SMOOTH_MS):
    print(f"\n*** time_R2 rises MONOTONICALLY with smoothing and its argmax sits on the")
    print(f"    boundary (sigma = {argmax_sigma:.0f} ms, the largest we swept). That is not a")
    print(f"    choice, it is a boundary -- and worse, it is close to CIRCULAR: a Gaussian")
    print(f"    kernel IMPOSES temporal autocorrelation, and 'time_R2' measures exactly how")
    print(f"    predictable time is from the smoothed signal. Maximising it just maximises")
    print(f"    the smoother. time_R2 CANNOT select the smoothing. ***")
print(f"\nchoice_AUC varies by only {auc_range:.3f} across the whole sweep -- it is flat, so it")
print("cannot select the configuration either. That flatness is itself the good news:")
print("**the choice signal is robust to bin size and smoothing**, so nothing downstream")
print("rests on this knob.")

# Chosen on external grounds instead, and stated openly:
#  * 50 ms bins  -- the rSLDS ground-truth in src/dynamics/rslds.py recovers real attractors
#    at 50 ms and shot-noise at 20 ms; and GPFA (section 5) fits tau = 107-283 ms, so the
#    dynamics are far slower than 50 ms bins and nothing is lost by using them.
#  * sigma = 40 ms -- comfortably below the FASTEST fitted GPFA timescale (~107 ms), so it
#    denoises without blurring the latent it is meant to reveal. 80-160 ms would start to
#    smear the fastest real dynamics, which is why the rising time_R2 there is a warning,
#    not a recommendation.
BIN_S, SIGMA_MS = 0.050, 40.0
print(f"\n-> using bin = {BIN_S*1000:.0f} ms, sigma = {SIGMA_MS:.0f} ms, chosen on external")
print("   grounds (rSLDS ground-truth + the GPFA timescales fitted in section 5), NOT by")
print("   argmaxing a metric that the knob itself inflates.")

## 3. The embedding zoo, with each method's own quality metric printed on it

`PCA` reports variance explained **in the axis labels**, as asked. The others report what
they actually have:

| method | native fit metric | comparable metric |
|---|---|---|
| PCA | explained variance per axis | trustworthiness, stress-1 |
| MDS | **Kruskal stress-1** (computed explicitly, not read off `mds.stress_`) | " |
| t-SNE | **KL divergence** | " |
| Isomap | **reconstruction error** | " |
| UMAP / Laplacian / PHATE | **none exists** | " |

There is no "variance explained" for UMAP. Quoting one would be an invention. What every
method *can* be scored on is **trustworthiness** (are the low-D neighbours real, or
fabricated?) and **stress-1** (global distance distortion) — computed identically for all
seven, so the local-vs-global trade-off is visible instead of hidden.

In [ ]:
from src.geometry.embed import run_embeddings, METHODS

N_SUB, N_SUB_MDS = 2000, 700
t0 = time.time()
embeds, points = {}, {}
for region in REGIONS:
    counts, bc = tensors[(region, BIN_S)]
    n_trials, n_bins, _ = counts.shape
    X = population_matrix(counts, bin_size_s=BIN_S, smooth_sigma_ms=SIGMA_MS)
    sub = rng.choice(X.shape[0], min(N_SUB, X.shape[0]), replace=False)
    points[region] = {"X": X, "sub": sub, "sub_mds": sub[:N_SUB_MDS],
                      "time": np.tile(bc, n_trials), "trial": np.repeat(np.arange(n_trials), n_bins)}
    embeds[region] = run_embeddings(X, sub, idx_mds=sub[:N_SUB_MDS], seed=SEED)
    print(f"[{region}] " + "  ".join(f"{m}:{embeds[region][m].seconds:.0f}s" for m in METHODS))
print(f"\nall embeddings: {time.time() - t0:.0f} s")

In [ ]:
for region in REGIONS:
    P = points[region]
    fig, axes = plt.subplots(1, len(METHODS), figsize=(3.0 * len(METHODS), 3.8))
    for ax, m in zip(axes, METHODS):
        e = embeds[region][m]
        idx = P["sub_mds"] if m == "MDS" else P["sub"]
        s = ax.scatter(e.coords[:, 0], e.coords[:, 1], c=P["time"][idx], cmap=SEQUENTIAL,
                       s=3, linewidths=0)
        xl, yl = e.axis_labels()
        ax.set(xlabel=xl, ylabel=yl, xticks=[], yticks=[])
        ax.set_title(f"{m}\n{e.caption()}", fontsize=8)
    cb = fig.colorbar(s, ax=axes, fraction=0.012, pad=0.01)
    cb.set_label("time from stimOn (s)")
    fig.suptitle(f"{region} — {BIN_S*1000:.0f} ms bins, sigma {SIGMA_MS:.0f} ms",
                 color=region_color(region), fontweight="bold")
    savefig(fig, PHASE, f"02_zoo_{region}")
    plt.show()

qual = pd.DataFrame([
    {"region": r, "method": m,
     "trustworthiness": round(embeds[r][m].quality["trustworthiness"], 3),
     "stress1": round(embeds[r][m].quality["stress1"], 3),
     "native": embeds[r][m].caption().split("\n")[0]}
    for r in REGIONS for m in METHODS])
qual.to_csv(OUT / "embedding_quality.csv", index=False)
print(qual.pivot(index="method", columns="region", values="trustworthiness").to_string())
print("\nHigh trustworthiness + high stress-1 = local structure kept, global geometry destroyed.")
print("That is the t-SNE/UMAP signature, and it is why they cannot support a claim about")
print("trajectories or distances — only about neighbourhoods.")

## 4. dPCA — the variance split the project actually asks for

PCA finds the dominant directions but cannot say *why* they vary. dPCA decomposes the
trial-averaged activity by **cause** first:

* **t** — condition-independent (time). Onset transients, arousal, movement. Usually
  dominates, and it is *not* evidence of anything decision-related. PC1 of an ordinary PCA
  is mostly this, which is precisely why plain PCA cannot answer our question.
* **s** — stimulus (side of the grating)
* **d** — decision (the animal's choice)
* **sd**, and the time-interactions — the mixtures

**The project's prediction, testable here for the first time: VISp should carry relatively
more *stimulus* variance, SCm/MRN more *decision* variance.**

The design must be balanced across stimulus x decision, and the (stim-left, chose-right)
cells are **error trials** — rare in a trained mouse. Every cell is therefore subsampled to
the smallest one, and that number is printed. If it is small, the decision components are
noisy and must be reported as such rather than quietly believed.

In [ ]:
from src.dynamics.dpca import (balanced_conditions, fit_dpca, variance_by_marginalization,
                               MARGINALIZATION_NAMES)

t0 = time.time()
dpca_var, dpca_Z, dpca_models = {}, {}, {}
for region in REGIONS:
    counts, bc = tensors[(region, BIN_S)]
    X, trialX, K = balanced_conditions(counts, stim_right, choice, seed=SEED)
    model, Z = fit_dpca(X, trialX, n_components=4)
    dpca_var[region] = variance_by_marginalization(model)
    dpca_Z[region], dpca_models[region] = Z, model
    print(f"[{region}] {X.shape[0]} units, K = {K} trials per (stimulus x decision) cell")
    print(dpca_var[region].to_string(index=False))
    print()
print(f"dPCA: {time.time() - t0:.0f} s")

In [ ]:
# --- the money plot: variance by cause, per region ---
keys = sorted({k for r in REGIONS for k in dpca_var[r]["marginalization"]})
fig, ax = plt.subplots(figsize=(10, 4.4))
w = 0.25
for i, region in enumerate(REGIONS):
    d = dpca_var[region].set_index("marginalization")["pct_of_explained"]
    v = [d.get(k, 0.0) for k in keys]
    ax.bar(np.arange(len(keys)) + (i - 1) * w, v, w, color=region_color(region),
           edgecolor="black", linewidth=0.5, label=region)
ax.set(xticks=range(len(keys)),
       xticklabels=[MARGINALIZATION_NAMES.get(k, k).replace(" (condition-independent)", "")
                    for k in keys],
       ylabel="% of explained variance", title="dPCA: what CAUSES the population variance?")
ax.tick_params(axis="x", rotation=20)
ax.legend(frameon=False)
fig.tight_layout(); savefig(fig, PHASE, "03_dpca_variance"); plt.show()

# the specific comparison the project predicts
comp = []
for region in REGIONS:
    d = dpca_var[region].set_index("marginalization")["pct_of_explained"]
    stim = sum(d.get(k, 0) for k in ["s", "st", "ts"])
    dec  = sum(d.get(k, 0) for k in ["d", "dt", "td"])
    comp.append({"region": region, "stimulus_%": round(stim, 1), "decision_%": round(dec, 1),
                 "time_%": round(d.get("t", 0), 1),
                 "decision_minus_stimulus": round(dec - stim, 1)})
comp = pd.DataFrame(comp)
comp.to_csv(OUT / "dpca_variance.csv", index=False)
print(comp.to_string(index=False))
print("\nPrediction: decision_minus_stimulus should be LOWEST in VISp and HIGHEST in SCm/MRN.")
print("Note how large time_% is everywhere: most of what a raw PCA trajectory shows is the")
print("condition-independent component, in EVERY area. That is the trap dPCA exists to avoid.")

In [ ]:
# --- dPCA component trajectories: stimulus and decision, per region ---
fig, axes = plt.subplots(len(REGIONS), 3, figsize=(12, 3.1 * len(REGIONS)), sharex=True)
bc = tensors[(REGIONS[0], BIN_S)][1]
ls = {0: "-", 1: "--"}          # decision: solid / dashed  (never colour-coded alone)
for i, region in enumerate(REGIONS):
    Z = dpca_Z[region]
    for j, key in enumerate(["t", "s", "d"]):
        ax = axes[i, j]
        if key not in Z:
            ax.set_axis_off(); continue
        for s in (0, 1):
            for d in (0, 1):
                ax.plot(bc, Z[key][0, s, d, :], ls=ls[d],
                        color=OKABE_ITO[4] if s == 0 else OKABE_ITO[0], lw=1.6,
                        path_effects=stroke(3.0))
        ax.axvline(0, color="black", ls=":", lw=1)
        if i == 0:
            ax.set_title(f"{key}: {MARGINALIZATION_NAMES.get(key, key)}", fontsize=9)
        if j == 0:
            ax.set_ylabel(region, color=region_color(region), fontweight="bold")
axes[-1, 1].set_xlabel("time from stimOn (s)")
fig.legend(handles=[Line2D([], [], color=OKABE_ITO[4], label="stimulus left"),
                    Line2D([], [], color=OKABE_ITO[0], label="stimulus right"),
                    Line2D([], [], color="0.3", ls="-", label="chose left"),
                    Line2D([], [], color="0.3", ls="--", label="chose right")],
           loc="lower center", ncol=4, frameon=False, fontsize=9)
fig.suptitle("dPCA component 1 of each marginalisation (colour = stimulus, line = choice)")
fig.tight_layout(rect=[0, 0.05, 1, 1]); savefig(fig, PHASE, "04_dpca_components"); plt.show()
print("A real stimulus component: the two COLOURS separate. A real decision component: the")
print("two LINE STYLES separate. If they separate before t=0, something is wrong (leakage).")

## 5. GPFA — the trajectory, with the smoothing *inferred* instead of imposed

Every embedding so far ran on data **we** smoothed. That is a hyper-parameter we chose, and
a smooth trajectory drawn from smoothed data is partly our own doing. GPFA removes the
objection: it models the latent as a Gaussian process across time and the spikes as noisy
observations, so it takes **raw counts** and *infers* the timescale.

The fitted `tau` per latent dimension is therefore a result in its own right — it is the
timescale the data say the population dynamics run at, and it can be compared directly
against the bin size we picked in §2.

In [ ]:
from src.dynamics.gpfa import trials_to_spiketrains, fit_gpfa, latent_timescales

X_DIM = 5
t0 = time.time()
gpfa_lat, gpfa_tau = {}, {}
for region in REGIONS:
    st = trials_to_spiketrains(trains[region], events, pre_s=PRE_S, post_s=POST_S)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model, lat = fit_gpfa(st, bin_size_s=BIN_S, x_dim=X_DIM, em_max_iters=50)
    gpfa_lat[region] = lat                                  # (trials, x_dim, bins)
    gpfa_tau[region] = latent_timescales(model, BIN_S)
    print(f"[{region}] latents {lat.shape} | fitted tau (ms): "
          f"{np.round(np.sort(gpfa_tau[region])[::-1], 1)}")
print(f"\nGPFA: {time.time() - t0:.0f} s")

tau_rows = []
for r in REGIONS:
    row = {"region": r}
    for i, t in enumerate(np.sort(gpfa_tau[r])[::-1]):
        row[f"tau{i+1}_ms"] = round(float(t), 1)
    tau_rows.append(row)
tau_tab = pd.DataFrame(tau_rows)
tau_tab.to_csv(OUT / "gpfa_timescales.csv", index=False)
print("\nIf the slowest tau is >> the bin size, the dynamics are slower than our bins and a")
print("coarser bin loses nothing. If tau ~ bin size, we are binning away the dynamics.")

In [ ]:
bc = tensors[(REGIONS[0], BIN_S)][1]
n_show = 40
fig, axes = plt.subplots(2, len(REGIONS), figsize=(4.4 * len(REGIONS), 7))
for j, region in enumerate(REGIONS):
    lat = gpfa_lat[region]
    idx = rng.choice(lat.shape[0], n_show, replace=False)

    # top: latent 1 vs time, single trials + mean
    ax = axes[0, j]
    for i in idx:
        ax.plot(bc, lat[i, 0], color=region_color(region), alpha=0.18, lw=0.8)
    ax.plot(bc, lat[:, 0].mean(0), color="black", lw=2.4, label="mean over trials")
    ax.axvline(0, color="black", ls=":", lw=1)
    ax.set(xlabel="time from stimOn (s)", ylabel="latent 1",
           title=f"{region}: GPFA latent 1 ({n_show} single trials)")
    ax.legend(frameon=False, fontsize=8)

    # bottom: 2-D latent trajectory, single trials, coloured by time
    ax = axes[1, j]
    for i in idx:
        ax.scatter(lat[i, 0], lat[i, 1], c=bc, cmap=SEQUENTIAL, s=4, linewidths=0, alpha=0.5)
    s = ax.scatter(lat[:, 0].mean(0), lat[:, 1].mean(0), c=bc, cmap=SEQUENTIAL,
                   s=45, edgecolors="black", linewidths=0.7, zorder=5)
    ax.set(xlabel="latent 1", ylabel="latent 2",
           title="single-trial latent trajectories\n(big dots = trial mean)")
fig.colorbar(s, ax=axes[1, :], fraction=0.02).set_label("time from stimOn (s)")
fig.suptitle("GPFA — inferred (not imposed) smoothness, fitted on RAW counts", fontweight="bold")
savefig(fig, PHASE, "05_gpfa_latents"); plt.show()

## 6. RRR — the communication subspace between areas

The HMM says *when* an area switches state; PCA says what its geometry looks like. Neither
says **what one area sends to another**. Reduced-rank regression does: predict area B's
population from area A's, restricted to rank r. Where the held-out R² saturates is the
**dimensionality of the channel** through which A influences B (Semedo et al. 2019).

**The control is everything.** "SCm predicts MRN at rank 3" means nothing alone — any two
populations correlate. The reference is the **within-area control**: split the *target*'s
units in half and predict one half from the other. If the cross-area rank is *lower* than
the within-area rank, the areas communicate through a genuinely restricted subspace. If they
are equal, there is no evidence of a selective channel.

CV is grouped **by trial** throughout, for the usual reason: bins of one trial are correlated
and an ungrouped split would leak.

In [ ]:
from src.dynamics.rrr import rrr_cv_curve, within_area_control, saturation_rank

mats = {r: population_matrix(tensors[(r, BIN_S)][0], bin_size_s=BIN_S, smooth_sigma_ms=SIGMA_MS)
        for r in REGIONS}
n_trials, n_bins = tensors[(REGIONS[0], BIN_S)][0].shape[:2]
groups = np.repeat(np.arange(n_trials), n_bins)

PAIRS = [("VISp", "SCm"), ("VISp", "MRN"), ("SCm", "MRN"), ("MRN", "SCm")]
RANKS = list(range(1, 11))

t0 = time.time()
curves, rrr_rows = {}, []
for src, tgt in PAIRS:
    c = rrr_cv_curve(mats[src], mats[tgt], groups, ranks=RANKS)
    curves[(src, tgt)] = c
    rrr_rows.append({"source": src, "target": tgt, "kind": "cross-area",
                     "best_r2": round(float(c["mean_r2"].max()), 3),
                     "sat_rank": saturation_rank(c)})
for tgt in REGIONS:
    c = within_area_control(mats[tgt], groups, ranks=RANKS, seed=SEED)
    curves[(tgt, tgt)] = c
    rrr_rows.append({"source": f"{tgt} (half)", "target": tgt, "kind": "within-area control",
                     "best_r2": round(float(c["mean_r2"].max()), 3),
                     "sat_rank": saturation_rank(c)})

rrr_tab = pd.DataFrame(rrr_rows)
rrr_tab.to_csv(OUT / "rrr_ranks.csv", index=False)
print(rrr_tab.to_string(index=False))
print(f"\nRRR: {time.time() - t0:.0f} s")

In [ ]:
fig, axes = plt.subplots(1, len(REGIONS), figsize=(5.0 * len(REGIONS), 4), sharey=True)
for ax, tgt in zip(axes, REGIONS):
    for src, t in PAIRS:
        if t != tgt:
            continue
        c = curves[(src, tgt)]
        ax.errorbar(c["rank"], c["mean_r2"], yerr=c["sem_r2"], marker="o", ms=4, lw=1.6,
                    color=region_color(src), label=f"{src} -> {tgt}", path_effects=stroke(3.0))
    c = curves[(tgt, tgt)]
    ax.errorbar(c["rank"], c["mean_r2"], yerr=c["sem_r2"], marker="s", ms=4, lw=1.8,
                color="black", ls="--", label=f"{tgt} half -> half (control)")
    ax.set(xlabel="rank of the regression", title=f"target: {tgt}")
    ax.legend(frameon=False, fontsize=8)
axes[0].set_ylabel("held-out R2 (grouped by trial)")
fig.suptitle("Communication subspace: how many dimensions does one area send to another?",
             fontweight="bold")
fig.tight_layout(); savefig(fig, PHASE, "06_rrr_communication"); plt.show()

print("Read it against the BLACK dashed control, never against zero:")
print("  cross-area saturates at a LOWER rank than within-area -> restricted channel (the")
print("     interesting outcome: B does not receive all of A's dimensions).")
print("  cross-area R2 far below the control -> the areas simply share little variance.")

## 7. One manifold, three areas

You cannot honestly stack three areas' point clouds and embed them: a VISp state vector
lives over 36 VISp neurons and an MRN one over 63 MRN neurons — **different spaces**. Stacked
and embedded, the areas would separate *trivially*, because their axes differ, and the plot
would be an artefact.

What licenses a common space here is that the three areas are recorded **simultaneously**:
they share the same (trial, bin) samples. So:

1. Concatenate all units into one population (VISp | SCm | MRN) and take its **joint PCA**.
   Those axes are the shared latent space — defined once, by the triplet as a whole.
2. Because PCA is **linear**, the joint trajectory decomposes *exactly* into a sum of
   per-area contributions, `Y(t) = Σ_a Y_a(t)`, where `Y_a` uses only area a's loadings.
   `Y_a` is **area a's contribution to the shared state, in coordinates every area shares**.
   The notebook asserts that identity rather than assuming it.

An area that *drives* the shared trajectory has a large, time-structured `Y_a`; one that
merely follows has a small or temporally flat one.

*Caveat kept in view:* areas with more units contribute more. Units are z-scored, so loudness
is controlled; unit **count** is not. The unit counts are printed with the variance shares.

In [ ]:
from src.geometry.joint import joint_matrix, joint_latent, partial_projections, area_drive

Xj, slices = joint_matrix(mats)                      # rows = the same (trial, bin) samples
pca_j, Yj = joint_latent(Xj, n_components=10, seed=SEED)
parts = partial_projections(Xj, pca_j, slices)       # asserts sum(parts) == Yj

drive = area_drive(parts)
drive["n_units"] = [mats[r].shape[1] for r in drive["region"]]
drive.to_csv(OUT / "joint_area_drive.csv", index=False)

v = pca_j.explained_variance_ratio_
print(f"joint population: {Xj.shape[1]} units, {Xj.shape[0]:,} samples")
print(f"joint PC1 = {100*v[0]:.1f}% var, PC2 = {100*v[1]:.1f}%, PC1-10 = {100*v[:10].sum():.1f}%\n")
print(drive.to_string(index=False))
print("\nvar_share = each area's share of the JOINT latent variance. These need not sum to 1:")
print("the areas' contributions are not orthogonal, and the remainder is cross-area")
print("covariance -- which is itself the shared signal the project is arguing about.")

In [ ]:
# --- Figure A: the common manifold, coloured by AREA ---
P = points[REGIONS[0]]
n_plot = 1500
pick = rng.choice(Yj.shape[0], n_plot, replace=False)
t_row = np.tile(tensors[(REGIONS[0], BIN_S)][1], n_trials)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.6))
xl = f"joint PC1 ({100*v[0]:.1f}% var)"
yl = f"joint PC2 ({100*v[1]:.1f}% var)"

ax = axes[0]
for region in REGIONS:
    Y = parts[region]
    ax.scatter(Y[pick, 0], Y[pick, 1], s=7, linewidths=0, alpha=0.55,
               color=region_color(region), label=f"{region} ({mats[region].shape[1]}u)")
ax.set(xlabel=xl, ylabel=yl, title="each area's CONTRIBUTION to the shared state")
ax.legend(frameon=False, markerscale=2)

# the joint state itself, for reference: it is the sum of the three clouds above
ax = axes[1]
s = ax.scatter(Yj[pick, 0], Yj[pick, 1], c=t_row[pick], cmap=SEQUENTIAL, s=7, linewidths=0)
ax.set(xlabel=xl, ylabel=yl, title="the joint state = sum of the three contributions")
fig.colorbar(s, ax=ax).set_label("time from stimOn (s)")
fig.suptitle("Common manifold across the triplet (joint PCA of all 3 areas' units)",
             fontweight="bold")
fig.tight_layout(); savefig(fig, PHASE, "07_common_manifold_by_area"); plt.show()

In [ ]:
# --- Figure B: SHAPE = area, COLOUR = time (as requested) ---
# Shape carries the area and colour carries time, so neither is doing double duty -- and the
# area distinction survives greyscale AND red-green colour-vision deficiency.
SHAPES = {"VISp": "o", "SCm": "^", "MRN": "s"}
n_plot2 = 900
pick2 = rng.choice(Yj.shape[0], n_plot2, replace=False)

fig, ax = plt.subplots(figsize=(8.2, 6.6))
for region in REGIONS:
    Y = parts[region]
    s = ax.scatter(Y[pick2, 0], Y[pick2, 1], c=t_row[pick2], cmap=SEQUENTIAL,
                   marker=SHAPES[region], s=26, linewidths=0.35, edgecolors="black", alpha=0.85)
ax.set(xlabel=xl, ylabel=yl,
       title="Common manifold — shape = area, colour = time from stimulus")
fig.colorbar(s, ax=ax).set_label("time from stimOn (s)")
ax.legend(handles=[Line2D([], [], ls="", marker=SHAPES[r], color="0.35",
                          markeredgecolor="black", label=r) for r in REGIONS],
          frameon=False, loc="best")
fig.tight_layout(); savefig(fig, PHASE, "08_common_manifold_shape_time"); plt.show()

print("What to look for: do the three areas trace the SAME path through the shared space")
print("(one manifold, area-specific amplitudes) or DIFFERENT paths (area-specific geometry)?")
print("That is the project's central claim, and this is the first figure that can address it.")

In [ ]:
# --- Figure C: the same common space, but NONLINEAR ---
# PHATE cannot partially project (only a linear map can), so the shared LINEAR basis is built
# first and PHATE is run on the stacked per-area contributions. The nonlinear step therefore
# acts on coordinates that are already comparable across areas -- which is what makes it legal.
import phate

n_each = 700
idx_e = rng.choice(Yj.shape[0], n_each, replace=False)
stack = np.concatenate([parts[r][idx_e] for r in REGIONS], axis=0)
area_of = np.concatenate([[r] * n_each for r in REGIONS])
time_of = np.tile(t_row[idx_e], len(REGIONS))

t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    Ye = phate.PHATE(n_components=2, knn=15, verbose=0, random_state=SEED,
                     n_jobs=-1).fit_transform(stack)
print(f"joint PHATE on stacked contributions: {time.time() - t0:.0f} s")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.6))
for region in REGIONS:
    m = area_of == region
    axes[0].scatter(Ye[m, 0], Ye[m, 1], s=8, linewidths=0, alpha=0.6,
                    color=region_color(region), label=region)
axes[0].set(xlabel="PHATE 1", ylabel="PHATE 2", xticks=[], yticks=[],
            title="colour = area")
axes[0].legend(frameon=False, markerscale=2)

for region in REGIONS:
    m = area_of == region
    s = axes[1].scatter(Ye[m, 0], Ye[m, 1], c=time_of[m], cmap=SEQUENTIAL,
                        marker=SHAPES[region], s=22, linewidths=0.3, edgecolors="black")
axes[1].set(xlabel="PHATE 1", ylabel="PHATE 2", xticks=[], yticks=[],
            title="shape = area, colour = time")
axes[1].legend(handles=[Line2D([], [], ls="", marker=SHAPES[r], color="0.35",
                               markeredgecolor="black", label=r) for r in REGIONS],
               frameon=False)
fig.colorbar(s, ax=axes[1]).set_label("time from stimOn (s)")
fig.suptitle("Nonlinear common manifold (PHATE on the shared linear basis)", fontweight="bold")
fig.tight_layout(); savefig(fig, PHASE, "09_common_manifold_phate"); plt.show()

print("PHATE has no variance-explained to quote -- it does not define one. Judge it by the")
print("trustworthiness in the table of section 3, and by whether it agrees with the linear")
print("picture above. Where they disagree, believe the LINEAR one: it cannot invent geometry.")

In [ ]:
lines = [
    "# Phase 3 — dPCA / GPFA / RRR / common manifold",
    "",
    f"Session `{SESSION_EID}`, regions {REGIONS}, {n_trials} trials.",
    f"Chosen configuration: **{BIN_S*1000:.0f} ms bins, sigma {SIGMA_MS:.0f} ms**, swept over "
    f"{[int(b*1000) for b in BIN_SIZES_S]} ms x {[int(s) for s in SMOOTH_MS]} ms.",
    "",
    "**The sweep cannot pick the smoothing, and we did not let it.** `time_R2` rises",
    "monotonically with sigma and its argmax lands on the boundary -- a Gaussian kernel",
    "imposes exactly the temporal autocorrelation that `time_R2` rewards, so maximising it",
    "maximises the smoother. `choice_AUC` is flat across the entire sweep, which is the real",
    "news: **the choice signal does not depend on bin size or smoothing at all.** The",
    "configuration was therefore chosen externally (rSLDS ground-truth bin size + the GPFA",
    "timescales below).",
    "",
    "## Bin size x smoothing (mean over regions and bins, by smoothing)",
    "",
    grand.to_markdown(),
    "",
    "## Embedding quality (there IS no variance-explained for UMAP/Laplacian/PHATE)",
    "",
    qual.to_markdown(index=False),
    "",
    "## dPCA — variance by cause",
    "",
    comp.to_markdown(index=False),
    "",
    "Project prediction: `decision_minus_stimulus` lowest in VISp, highest in SCm/MRN.",
    "",
    "## GPFA — fitted latent timescales (ms)",
    "",
    tau_tab.to_markdown(index=False),
    "",
    "## RRR — communication subspace (vs within-area control)",
    "",
    rrr_tab.to_markdown(index=False),
    "",
    "## Joint manifold — each area's share of the shared latent",
    "",
    drive.to_markdown(index=False),
    "",
    f"Joint PC1 = {100*v[0]:.1f}% var, PC2 = {100*v[1]:.1f}%, PC1-10 = {100*v[:10].sum():.1f}%.",
    "",
    f"Figures: `results/figures/{PHASE}/`.",
]
(OUT / "summary.md").write_text("\n".join(lines), encoding="utf-8")
print((OUT / "summary.md").read_text(encoding="utf-8"))